# E-Tickets · Clásico Nacional · Actividad 2 · Yoni Quirán

**MVP con TDD/BDD:** plataforma de boletos para el partido Municipal vs. Comunicaciones, con selección de localidades y asientos.

El proyecto conserva las 7 pruebas de aceptación proporcionadas por la docente y agrega 14 casos unitarios propios. La lógica obligatoria se encuentra en `venta_entradas/modelos.py`.

## 1. Ejecutar la aplicación E-Tickets

Ejecuta la siguiente celda con `Shift + Enter`. Cuando aparezca el mensaje de confirmación, abre `http://127.0.0.1:8765` en el navegador.

Datos para el formulario: tarjeta **4242 4242 4242 4242**, vencimiento **12/30** y CVV **123**.

In [1]:
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from threading import Thread, Lock
from urllib.parse import urlparse
from datetime import date
from uuid import uuid4
import json


class Taquilla:
    """Reglas de negocio del MVP, independientes de la pantalla."""

    LOCALIDADES = {
        "Palco": 550,
        "Tribuna": 350,
        "Preferencia": 250,
        "General Norte": 150,
        "General Sur": 150,
    }

    def __init__(self):
        self._lock = Lock()
        self.asientos = {}
        for prefijo, (localidad, filas, columnas) in {
            "P": ("Palco", 2, 4),
            "T": ("Tribuna", 2, 6),
            "F": ("Preferencia", 2, 7),
            "N": ("General Norte", 2, 6),
            "S": ("General Sur", 2, 6),
        }.items():
            for fila in range(1, filas + 1):
                for numero in range(1, columnas + 1):
                    codigo = f"{prefijo}{fila}-{numero}"
                    self.asientos[codigo] = {
                        "id": codigo,
                        "localidad": localidad,
                        "precio": self.LOCALIDADES[localidad],
                    }
        self.vendidos = {"P1-1", "T1-3", "F2-4", "N1-5", "S2-2"}
        self.boletos = {}

    def listar_asientos(self):
        return [
            {**asiento, "disponible": codigo not in self.vendidos}
            for codigo, asiento in self.asientos.items()
        ]

    @staticmethod
    def validar_tarjeta(numero, vencimiento, cvv):
        digitos = "".join(caracter for caracter in numero if caracter.isdigit())
        if digitos != "4242424242424242":
            raise ValueError("Utiliza la tarjeta de prueba 4242 4242 4242 4242.")
        try:
            mes, anio = map(int, vencimiento.split("/"))
            vigente = 1 <= mes <= 12 and (anio, mes) >= (date.today().year % 100, date.today().month)
        except (ValueError, AttributeError):
            vigente = False
        if not vigente:
            raise ValueError("La fecha de vencimiento no es válida.")
        if not (cvv.isdigit() and len(cvv) == 3):
            raise ValueError("El CVV debe tener tres dígitos.")

    def comprar(self, codigos, comprador, tarjeta, vencimiento, cvv):
        if not comprador.strip():
            raise ValueError("Escribe el nombre del comprador.")
        if not codigos:
            raise ValueError("Selecciona al menos un asiento.")
        if len(codigos) > 6:
            raise ValueError("Puedes comprar un máximo de seis boletos.")
        if len(set(codigos)) != len(codigos):
            raise ValueError("La selección contiene asientos repetidos.")
        self.validar_tarjeta(tarjeta, vencimiento, cvv)
        with self._lock:
            inexistentes = [codigo for codigo in codigos if codigo not in self.asientos]
            no_disponibles = [codigo for codigo in codigos if codigo in self.vendidos]
            if inexistentes:
                raise ValueError("Uno de los asientos seleccionados no existe.")
            if no_disponibles:
                raise ValueError("Uno de los asientos ya no está disponible.")
            self.vendidos.update(codigos)
            detalle = [self.asientos[codigo] for codigo in codigos]
            compra = {
                "codigo": f"CL-{uuid4().hex[:10].upper()}",
                "evento": "Clásico Nacional: Rojos vs. Cremas",
                "fecha": "22 de noviembre de 2026 · 18:00",
                "estadio": "Estadio Nacional Doroteo Guamuch Flores",
                "comprador": comprador.strip(),
                "asientos": detalle,
                "total": sum(item["precio"] for item in detalle),
            }
            self.boletos[compra["codigo"]] = compra
            return compra


HTML = r'''<!doctype html>
<html lang="es"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Clásico Nacional · E-Tickets </title><style>
:root{--ink:#101114;--paper:#f5f3ef;--card:#fff;--gold:#bd9654;--red:#a71930;--cream:#f1ede2;--muted:#77736b;--line:#ded9d0}*{box-sizing:border-box}body{margin:0;background:var(--paper);color:var(--ink);font:14px/1.5 Inter,Arial,sans-serif}header{height:68px;padding:0 max(5vw,24px);display:flex;align-items:center;justify-content:space-between;background:#101114;color:#fff}.brand{font-size:20px;font-weight:800;letter-spacing:1px}.brand i{color:var(--gold);font-style:normal}.safe{font-size:12px;color:#d8d2c8}main{max-width:1120px;margin:auto;padding:28px 22px 60px}.hero{position:relative;overflow:hidden;min-height:240px;border-radius:22px;padding:38px;background:linear-gradient(115deg,#17181c 0 47%,#7b1224 48% 67%,#ded4bd 68%);color:#fff;display:flex;justify-content:space-between;align-items:end;box-shadow:0 18px 50px #16161622}.eyebrow{color:#d9b979;letter-spacing:2px;font-size:11px;font-weight:700}.hero h1{max-width:620px;font-size:42px;line-height:1.03;margin:10px 0}.hero p{color:#dedbd4}.versus{font-size:70px;font-weight:900;opacity:.18}.steps{display:flex;justify-content:center;gap:46px;padding:22px;color:#918c83;font-size:12px}.steps b{color:var(--ink)}.layout{display:grid;grid-template-columns:1.45fr .75fr;gap:20px}.panel{background:var(--card);border:1px solid var(--line);border-radius:18px;padding:24px;box-shadow:0 8px 28px #11111108}h2{font-size:21px;margin:0 0 5px}.sub{color:var(--muted);margin:0 0 18px}.localidad{margin:16px 0 6px;display:flex;justify-content:space-between;align-items:center;font-weight:700}.localidad span{font-size:12px;color:var(--muted);font-weight:400}.field{height:48px;border:2px solid #d8bf88;border-radius:60% 60% 8px 8px;margin:10px 25px 22px;display:grid;place-items:center;color:#9b824d;font-size:11px;letter-spacing:2px}.seats{display:grid;grid-template-columns:repeat(7,38px);gap:8px;justify-content:center}.seat{height:34px;border:1px solid #d7d2ca;background:#f4f2ee;border-radius:8px;font-size:10px;cursor:pointer;color:#4c4944}.seat:hover{border-color:var(--gold)}.seat.selected{background:#15161a;color:#fff;border-color:#15161a}.seat.sold{background:#e5e2dc;color:#aaa49b;text-decoration:line-through;cursor:not-allowed}.legend{display:flex;gap:18px;justify-content:center;margin-top:20px;color:var(--muted);font-size:11px}.summaryBox{background:#f7f5f1;border-radius:12px;padding:14px;min-height:62px;margin:18px 0}.item{display:flex;justify-content:space-between;padding:5px 0;border-bottom:1px solid #e9e4dc}.total{display:flex;justify-content:space-between;font-size:18px;font-weight:800;padding-top:12px}label{display:block;font-size:12px;font-weight:700;margin-top:12px}input{width:100%;margin-top:5px;border:1px solid #d7d2c9;border-radius:9px;padding:11px;font:inherit;background:#fff}.row{display:grid;grid-template-columns:1fr 1fr;gap:10px}.buy,.print{width:100%;border:0;border-radius:10px;padding:13px;background:#141519;color:#fff;font-weight:700;margin-top:18px;cursor:pointer}.buy:disabled{opacity:.35}.note{font-size:10px;text-align:center;color:var(--muted)}#message{color:#a71930;font-size:12px}.ticket{margin-top:20px;background:#111216;color:#fff;border-radius:18px;padding:28px;display:grid;grid-template-columns:1fr 120px 180px;gap:25px;align-items:center}.ticket small{color:#d9b979;letter-spacing:1.5px}.ticket h2{font-size:26px}.qr{width:102px;height:102px;border:8px solid #fff;background:repeating-conic-gradient(#111 0 25%,#fff 0 50%) 0/18px 18px}.hidden{display:none}footer{text-align:center;padding:25px;color:var(--muted);font-size:11px}@media(max-width:780px){.layout{grid-template-columns:1fr}.hero h1{font-size:31px}.versus{display:none}.steps{gap:18px}.ticket{grid-template-columns:1fr}.seats{grid-template-columns:repeat(6,38px)}}@media print{header,.hero,.steps,.layout,footer,.print{display:none!important}.ticket{display:grid!important;margin:20px}}
/* Identidad final: rojo intenso de Municipal y blanco de Comunicaciones */
body{background:#f4f4f2}header{background:#d70b20;border-bottom:3px solid #fff}.brand i{color:#fff}.hero{min-height:270px;padding:0;background:linear-gradient(90deg,#d70b20 0 50%,#fff 50% 100%);display:grid;grid-template-columns:1fr 210px 1fr;align-items:stretch;box-shadow:0 18px 50px #17000522}.team{display:flex;align-items:center;justify-content:center;gap:22px;padding:26px}.team img{width:128px;height:128px;object-fit:contain;filter:drop-shadow(0 10px 12px #0003)}.team-name{font-size:28px;font-weight:900;letter-spacing:.5px}.team-label{font-size:10px;letter-spacing:2px;font-weight:800;opacity:.75}.rojos{color:#fff}.cremas{color:#17181c}.match-center{display:flex;flex-direction:column;align-items:center;justify-content:center;text-align:center;background:#17181c;color:#fff;clip-path:polygon(18% 0,100% 0,82% 100%,0 100%);padding:22px 12px}.match-center strong{font-size:44px;line-height:1}.match-center span{color:#ddd;font-size:10px;letter-spacing:1.5px}.match-center b{color:#fff;font-size:13px;margin-bottom:12px}.match-center small{color:#ddd;font-size:9px;margin-top:12px}.seat.selected{background:#d70b20;border-color:#d70b20}.buy,.print{background:#d70b20}.ticket{background:linear-gradient(115deg,#17181c 0 70%,#d70b20 70%)}h2{color:#17181c}.ticket h2{color:#fff}.field{border-color:#d70b20;color:#b20b1d}.localidad span{color:#9b1020}@media(max-width:780px){.hero{grid-template-columns:1fr 100px 1fr;min-height:220px}.team{flex-direction:column;gap:7px;padding:14px}.team img{width:74px;height:74px}.team-name{font-size:18px}.match-center strong{font-size:28px}.match-center b,.match-center small{display:none}}
</style></head><body><header><div class="brand">E<i>·</i>TICKETS</div></header><main>
<section class="hero"><div class="team rojos"><img src="..." alt="Escudo de Municipal"><div><div class="team-label">LOS ROJOS</div><div class="team-name">MUNICIPAL</div></div></div><div class="match-center"><b>CLÁSICO NACIONAL</b><strong>VS</strong><span>22 NOV 2026 · 18:00</span><small>ESTADIO DOROTEO GUAMUCH FLORES</small></div><div class="team cremas"><div><div class="team-label">LOS CREMAS</div><div class="team-name">COMUNICACIONES</div></div><img src="..." alt="Escudo de Comunicaciones"></div></section>
<div class="steps"><b>01 Localidad y asientos</b><span>02 Pago</span><span>03 Boletos</span></div>
<section id="compra" class="layout"><article class="panel"><h2>Selecciona tus asientos</h2><p class="sub">Hasta 6 boletos por compra.</p><div class="field">CANCHA</div><div id="mapa"></div><div class="legend"><span>□ Disponible</span><span>■ Seleccionado</span><span>▧ Ocupado</span></div></article>
<aside class="panel"><h2>Tu selección</h2><div id="resumen" class="summaryBox">Aún no has seleccionado asientos.</div><form id="pago"><label>Nombre del comprador<input id="comprador" required placeholder="Nombre y apellido"></label><label>Tarjeta de prueba<input id="tarjeta" required placeholder="4242 4242 4242 4242"></label><div class="row"><label>Vencimiento<input id="vencimiento" required placeholder="12/30"></label><label>CVV<input id="cvv" required placeholder="123"></label></div><button class="buy" id="comprar" disabled>Confirmar compra</button></form><p id="message"></p></aside></section>
<section id="boleto" class="ticket hidden"><div><small>COMPRA CONFIRMADA</small><h2>Tus boletos están listos</h2><p id="info"></p></div><div class="qr"></div><button class="print" onclick="print()">Imprimir boletos</button></section></main>
<script>
let elegidos=[];let catalogo=[];const q=n=>new Intl.NumberFormat('es-GT',{style:'currency',currency:'GTQ'}).format(n);
async function cargar(){catalogo=await fetch('/api/asientos').then(r=>r.json());const mapa=document.querySelector('#mapa');mapa.innerHTML='';const grupos={};catalogo.forEach(a=>(grupos[a.localidad]??=[]).push(a));Object.entries(grupos).forEach(([loc,lista])=>{const t=document.createElement('div');t.className='localidad';t.innerHTML=`${loc}<span>desde ${q(lista[0].precio)}</span>`;mapa.append(t);const caja=document.createElement('div');caja.className='seats';lista.forEach(a=>{const b=document.createElement('button');b.className=`seat ${a.disponible?'':'sold'}`;b.disabled=!a.disponible;b.textContent=a.id;b.title=`${a.localidad} · ${q(a.precio)}`;b.onclick=()=>seleccionar(a,b);caja.append(b)});mapa.append(caja)})}
function seleccionar(a,b){if(elegidos.some(x=>x.id===a.id)){elegidos=elegidos.filter(x=>x.id!==a.id);b.classList.remove('selected')}else{if(elegidos.length===6){message.textContent='Máximo seis boletos.';return}elegidos.push(a);b.classList.add('selected')}message.textContent='';actualizar()}
function actualizar(){comprar.disabled=!elegidos.length;if(!elegidos.length){resumen.textContent='Aún no has seleccionado asientos.';return}resumen.innerHTML=elegidos.map(a=>`<div class="item"><span>${a.id} · ${a.localidad}</span><b>${q(a.precio)}</b></div>`).join('')+`<div class="total"><span>Total</span><span>${q(elegidos.reduce((s,a)=>s+a.precio,0))}</span></div>`}
pago.onsubmit=async e=>{e.preventDefault();const data={asientos:elegidos.map(a=>a.id),comprador:comprador.value,tarjeta:tarjeta.value,vencimiento:vencimiento.value,cvv:cvv.value};const r=await fetch('/api/comprar',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify(data)});const x=await r.json();if(!r.ok){message.textContent=x.error;await cargar();return}compra.classList.add('hidden');boleto.classList.remove('hidden');info.innerHTML=`<b>${x.evento}</b><br>${x.fecha}<br>${x.estadio}<br><br>Comprador: ${x.comprador}<br>Asientos: ${x.asientos.map(a=>a.id+' ('+a.localidad+')').join(', ')}<br>Total: ${q(x.total)}<br>Código: <b>${x.codigo}</b>`;document.querySelectorAll('.steps>*').forEach((x,i)=>x.style.color=i===2?'#101114':'#918c83')};cargar();
</script></body></html>'''


TAQUILLA = Taquilla()


class Aplicacion(BaseHTTPRequestHandler):
    def responder(self, estado, contenido, tipo="application/json; charset=utf-8"):
        if isinstance(contenido, (dict, list)):
            contenido = json.dumps(contenido, ensure_ascii=False)
        datos = contenido.encode("utf-8")
        self.send_response(estado)
        self.send_header("Content-Type", tipo)
        self.send_header("Content-Length", str(len(datos)))
        self.end_headers()
        self.wfile.write(datos)

    def do_GET(self):
        ruta = urlparse(self.path).path
        if ruta == "/":
            return self.responder(200, HTML, "text/html; charset=utf-8")
        if ruta == "/api/asientos":
            return self.responder(200, TAQUILLA.listar_asientos())
        self.responder(404, {"error": "Recurso no encontrado."})

    def do_POST(self):
        if urlparse(self.path).path != "/api/comprar":
            return self.responder(404, {"error": "Recurso no encontrado."})
        try:
            longitud = int(self.headers.get("Content-Length", "0"))
            datos = json.loads(self.rfile.read(longitud) or b"{}")
            compra = TAQUILLA.comprar(
                datos.get("asientos", []), datos.get("comprador", ""),
                datos.get("tarjeta", ""), datos.get("vencimiento", ""), datos.get("cvv", "")
            )
            self.responder(201, compra)
        except (ValueError, json.JSONDecodeError) as error:
            self.responder(400, {"error": str(error)})

    def log_message(self, formato, *argumentos):
        pass


try:
    servidor.shutdown()
    servidor.server_close()
except NameError:
    pass

servidor = ThreadingHTTPServer(("127.0.0.1", 8765), Aplicacion)
hilo_servidor = Thread(target=servidor.serve_forever, daemon=True)
hilo_servidor.start()

print("Aplicación iniciada correctamente.")
print("Abre esta dirección en tu navegador: http://127.0.0.1:8765")


Aplicación iniciada correctamente.
Abre esta dirección en tu navegador: http://127.0.0.1:8765


## 2. Ejecutar las pruebas oficiales y unitarias

Esta celda usa `pytest` sobre la estructura solicitada por la docente:

- `tests/test_aceptacion.py`: 7 pruebas oficiales sin modificar.
- `tests/test_unitarias.py`: 14 casos propios desarrollados con TDD.

El resultado esperado es **21 passed**.

In [2]:
from pathlib import Path
import subprocess
import sys


raiz_proyecto = Path.cwd()
if not (raiz_proyecto / "pytest.ini").exists():
    raise FileNotFoundError(
        "Abre el notebook desde la carpeta principal del proyecto."
    )

comando = [sys.executable, "-m", "pytest", "-v", "--color=no"]
resultado = subprocess.run(
    comando,
    cwd=raiz_proyecto,
    capture_output=True,
    text=True,
)

print(resultado.stdout)
if resultado.stderr:
    print(resultado.stderr)

assert resultado.returncode == 0, "Hay pruebas que todavía no pasan."


============================= test session starts ==============================
platform darwin -- Python 3.13.9, pytest-8.4.2, pluggy-1.5.0 -- /opt/anaconda3/bin/python
cachedir: .pytest_cache
rootdir: /Users/yoniquiran/Documents/Codex/2026-08-31/ne/work/Actividad2_YONI_QUIRAN_actualizada
configfile: pytest.ini
testpaths: tests
plugins: anyio-4.10.0
collecting ... collected 21 items

tests/test_aceptacion.py::test_aceptacion_crear_evento_inicia_sin_entradas_vendidas PASSED [  4%]
tests/test_aceptacion.py::test_aceptacion_vender_entradas_dentro_del_aforo PASSED [  9%]
tests/test_aceptacion.py::test_aceptacion_no_se_puede_vender_mas_entradas_que_el_aforo PASSED [ 14%]
tests/test_aceptacion.py::test_aceptacion_no_se_puede_vender_a_un_evento_inexistente PASSED [ 19%]
tests/test_aceptacion.py::test_aceptacion_no_se_puede_vender_cantidad_invalida PASSED [ 23%]
tests/test_aceptacion.py::test_aceptacion_calculo_de_total_con_descuento PASSED [ 28%]
tests/test_aceptacion.py::test_aceptacion_de

## 3. Detener la aplicación

Ejecuta esta última celda cuando hayas terminado de usar el sitio.

In [3]:
servidor.shutdown()
servidor.server_close()
print('Aplicación detenida.')

Aplicación detenida.
